# 1. DiffSCMオーケストラの実装

## プログラムの出力確認や調整

In [1]:
import os

os.chdir("/home/hashikami/projects/Diff")

from pathlib import Path
from pipeline.utils import dict2namespace
import yaml


config_path = Path("config/config_diffscm.yaml")
with open(config_path, "r") as f:
    config_raw = yaml.load(f, Loader=yaml.FullLoader)
config = dict2namespace(config_raw)

anticausal = config.anticausal_predictor_training
key = 'thickness'
file_name = anticausal.checkpoint_dir + anticausal.checkpoint_name.key

AttributeError: 'Namespace' object has no attribute 'checkpoint_name'

In [11]:
from diffusion.utils.utils import load_json


attrs = load_json(config.image_data.meta_data.attribute_size_path)
for key in attrs.keys():
    # file名にkeyが含まれている場合にはそのファイル名を出力
    file_name = next((file for file in os.listdir(config.anticausal_predictor_training.checkpoint_dir) if key in file), None)
    print(file_name)

checkpoint_thickness_2025-11-28_18.pth
checkpoint_intensity_2025-11-28_19.pth
checkpoint_slant_2025-11-28_20.pth
checkpoint_width_2025-11-28_21.pth


In [18]:
import torch

from diffusion.utils.utils import load_json
from diffusion.classifier_guidance.classifier import AntiCausalPredictor, EncoderUNet
from diffusion.utils import sampling_utils

import importlib
importlib.reload(sampling_utils)

from diffusion.utils.sampling_utils import get_models_functions

causal_graph = load_json(config.image_data.meta_data.graph_path)
anticausal = config.anticausal_predictor_training
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

anti_causal_predictors = {attr: AntiCausalPredictor(encoder=EncoderUNet(cdim=len(causal_graph[attr]))) for attr in causal_graph.keys()}
for key , predictor in anti_causal_predictors.items():
    file_name = next((file for file in os.listdir(anticausal.checkpoint_dir) if key in file), None)
    predictor.load_state_dict(torch.load(anticausal.checkpoint_dir + file_name , map_location=device)["predictor_state_dict"])
    predictor.to(device)
    
cond_fn = get_models_functions(config, anti_causal_predictors)


In [21]:
x = torch.randn(256, 1, 28, 28).to(device)
t = torch.randint(0, 500, (256,), device=device).long()
cond = torch.randn(256, 4).to(device)

grad = cond_fn(x, t, cond)
grad.shape

torch.Size([256, 1, 28, 28])

In [15]:
import torch.nn.functional as F

a = torch.randn(256, 1)
b = torch.randn(256, 1)

c = F.mse_loss(a, b, reduction='none')
d = -0.5 * c.sum(dim=1)
d.shape

torch.Size([256])

In [17]:
a[:, 0].shape

torch.Size([256])